# DBA5115 Workshop Guide: Building Your Own Agents

This notebook walks you through setting up the agent platform and building two agents:
1. **Haiku Compose Agent** - Timer-triggered, no tools, purely prompt-driven
2. **Knowledge QA Agent** - Uses `file_search` over uploaded documents (RAG)

---

# Part 1: Environment Setup

## 1.1 VS Code Extensions

Open VS Code and press `Cmd+Shift+X` to open the Extensions panel. Install each of the following:

### Required

| Extension | Extension ID | Why |
|---|---|---|
| Python | `ms-python.python` | Python language support, IntelliSense, venv detection |
| Azure Functions | `ms-azuretools.vscode-azurefunctions` | Local Functions runtime, debugging, deployment |
| Pylance | `ms-python.vscode-pylance` | Type checking and autocomplete for Python |
| Azure Account | `ms-vscode.azure-account` | Azure sign-in (required by Azure Functions extension) |

### Recommended

| Extension | Extension ID | Why |
|---|---|---|
| REST Client | `humao.rest-client` | Quick HTTP testing without leaving VS Code |
| SQLTools | `mtxr.sqltools` | Browse SQL Server tables directly |

### How to verify
1. Open the Command Palette (`Cmd+Shift+P`)
2. Type `Python: Select Interpreter` - you should see your `.venv` listed, select it
3. Type `Azure Functions: Create New Project` - if this command appears, the extension is working (press **Escape**, don't actually create one)

## 1.2 Python Virtual Environment

Open the integrated terminal in VS Code (`` Ctrl+` ``) and run these commands:

In [ ]:
# Run these in the VS Code terminal (not in this notebook):
# python3 -m venv .venv
# source .venv/bin/activate
# pip install -r requirements.txt

# Verify: you should see (.venv) in your terminal prompt

## 1.3 Import Postman Collection

1. Open **Postman** (download from [postman.com](https://www.postman.com/downloads/) if not installed)
2. Click **Import** (top-left corner)
3. Drag and drop the file: `postman_collection.json` from the project root
4. A collection called **"DBA5115 Agent Platform API"** should appear in the sidebar
5. Expand it - you should see these folders:
   - Health
   - Hooks
   - Agent Definitions
   - Knowledge Files
   - Prompts
   - Tool Mappings
   - Seed

### Set the collection variables
1. Click the collection name **"DBA5115 Agent Platform API"** in the sidebar
2. Go to the **Variables** tab
3. Confirm these variables exist:

| Variable | Initial Value | Description |
|---|---|---|
| `baseUrl` | `http://localhost:7071` | Local Functions URL |
| `functionKey` | *(leave empty for local dev)* | Auth key (not needed locally) |
| `agentId` | *(auto-populated)* | Saved automatically when you call Create Agent |
| `promptId` | *(auto-populated)* | Saved automatically when you call Create Prompt |
| `toolId` | *(auto-populated)* | Saved automatically when you call Create Tool |
| `filename` | *(manual)* | Used for knowledge file deletion |

4. Click **Save** (`Cmd+S`)

## 1.4 Configure Environment Variables

### Step 1: Create the file
Copy the example file to create your local settings:

In [ ]:
# Run in terminal - creates local.settings.json from the example template
!cp local.settings.json.example local.settings.json
print("local.settings.json created successfully")

### Step 2: Fill in the values

Open `local.settings.json` in VS Code and replace **every** placeholder with your actual values:

```json
{
  "IsEncrypted": false,
  "Values": {
    "FUNCTIONS_WORKER_RUNTIME": "python",
    "AzureWebJobsStorage": "<paste Azure Storage connection string>",
    "AzureWebJobsFeatureFlags": "EnableWorkerIndexing",

    "SERVICE_BUS_CONNECTION_STRING": "<paste from Azure portal>",

    "AZURE_TENANT_ID": "<paste from Azure portal>",
    "AZURE_CLIENT_ID": "<paste from Azure portal>",
    "AZURE_CLIENT_SECRET": "<paste from Azure portal>",
    "AZURE_AI_ENDPOINT": "<paste from Azure portal>",

    "DB_SERVER": "<your-server>.database.windows.net",
    "DB_DATABASE": "<your-database>",
    "DB_USERNAME": "<your-username>",
    "DB_PASSWORD": "<your-password>",

    "GMAIL_CLIENT_ID": "<from Google Cloud Console>",
    "GMAIL_CLIENT_SECRET": "<from Google Cloud Console>",
    "GMAIL_REFRESH_TOKEN": "<from OAuth flow>",

    "NUS_EMAIL": "<your-email-to-monitor>",

    "GMAIL_FROM_NAME": "<display name for sent emails>",
    "GMAIL_FROM_EMAIL": "<your-gmail@gmail.com>",

    "DEFAULT_AGENT_MODEL": "gpt-4o-mini",

    "AGENT_CONFIG_BLOB_CONN_STR": "<paste from Azure portal>",
    "AGENT_CONFIG_CONTAINER": "agent-config",

    "FUNC_BASE_URL": "http://localhost:7071/api",
    "FUNC_KEY": ""
  }
}
```

### Where to find each value

| Variable | Where to find it |
|---|---|
| `AzureWebJobsStorage` | Azure Portal > Storage Account > Access keys > Connection string |
| `SERVICE_BUS_CONNECTION_STRING` | Azure Portal > Service Bus namespace > Shared access policies > RootManageSharedAccessKey > Primary Connection String |
| `AZURE_TENANT_ID` | Azure Portal > Microsoft Entra ID > Overview > Tenant ID |
| `AZURE_CLIENT_ID` | Azure Portal > App Registrations > your app > Application (client) ID |
| `AZURE_CLIENT_SECRET` | Azure Portal > App Registrations > your app > Certificates & secrets > Client secret value |
| `AZURE_AI_ENDPOINT` | Azure Portal > Azure AI Foundry > your project > Overview > Endpoint |
| `DB_SERVER` / `DB_DATABASE` / `DB_USERNAME` / `DB_PASSWORD` | Azure Portal > SQL Database > your database > Connection strings |
| `GMAIL_*` values | Google Cloud Console > APIs & Services > Credentials (see Gmail setup guide) |
| `AGENT_CONFIG_BLOB_CONN_STR` | Azure Portal > Storage Account > Access keys > Connection string |

> **IMPORTANT:** `local.settings.json` is in `.gitignore` - it will never be committed. Never share this file.

## 1.5 Start the Application

### Option A: VS Code Debugger (recommended)
1. Press `F5` or go to **Run > Start Debugging**
2. VS Code will:
   - Run `pip install -r requirements.txt` (build task)
   - Start the Azure Functions host on port `7071`
   - Attach the Python debugger on port `9091`
3. Watch the terminal for: `Host started` and the list of registered functions

### Option B: Terminal
```bash
func start
```

### What happens on startup
- SQL tables are created automatically (`AgentDefinition`, `AgentPromptRegistry`, `AgentToolMapping`, `LLMTokenUsage`)
- Blob container `agent-config` is created if it doesn't exist
- Default agents (`email_triage`, `notification_content`) are seeded into the database
- Default prompts and tool definitions are uploaded to Blob Storage

## 1.6 Verify the App is Running

Set up the base URL and run the health check. All API calls in this notebook use this variable.

In [ ]:
import requests
import json

BASE_URL = "http://localhost:7071"

# Store IDs as we create resources (mirrors Postman collection variables)
agent_id = None
prompt_id = None
tool_id = None

In [ ]:
# Health Check
r = requests.get(f"{BASE_URL}/api/health")
print(f"Status: {r.status_code}")
print(json.dumps(r.json(), indent=2))

# Expected:
# {
#   "status": "healthy",
#   "app": "dba5115-agent-template",
#   "layers": ["hooks", "agents"]
# }

## 1.7 Verify the Seed Worked

The app auto-seeds 2 default agents on first startup. Let's confirm they exist.

In [ ]:
# List all agents
r = requests.get(f"{BASE_URL}/api/config/agents")
agents = r.json()
print(f"Found {len(agents)} agents:\n")
for a in agents:
    print(f"  [{a['id']}] {a['name']} - {a['description']}")

# Expected: 2 agents - email_triage and notification_content

---
# Part 2: Build the Haiku Agent

The haiku agent composes haikus based on the current time and routes them to the notification agent for email delivery.

**Characteristics:**
- Triggered by a **timer** (every 1 minute)
- **No tools** - purely prompt-driven
- Even minute = nature haiku, odd minute = Singlish haiku
- Routes output to `notification_content` agent via `next_action`

## 2.1 Create the Agent Definition

Register the haiku agent in the database. No `knowledge_source` needed since this agent doesn't use RAG.

In [ ]:
# Create the haiku_compose agent
r = requests.post(f"{BASE_URL}/api/config/agents", json={
    "name": "haiku_compose",
    "description": "Haiku composer agent - creates time-based haikus and routes to notification",
    "model": "gpt-4o-mini"
})

print(f"Status: {r.status_code}")
result = r.json()
print(json.dumps(result, indent=2))

# Save the agent ID for subsequent requests
agent_id = result.get("id")
print(f"\nSaved agent_id = {agent_id}")

In [ ]:
# Verify: list all agents - should now show 3
r = requests.get(f"{BASE_URL}/api/config/agents")
agents = r.json()
print(f"Total agents: {len(agents)}\n")
for a in agents:
    print(f"  [{a['id']}] {a['name']}")

## 2.2 Upload the System Prompt

The instruction file already exists in the repo at `agents/instructions/haiku_compose.system.md`.

We upload it via the API as a **file upload** (multipart form-data). The platform stores it in Azure Blob Storage and links it to the agent in the database.

In [ ]:
# Let's first look at what the prompt says
with open("agents/instructions/haiku_compose.system.md", "r") as f:
    print(f.read())

In [ ]:
# Upload the prompt file
with open("agents/instructions/haiku_compose.system.md", "rb") as f:
    r = requests.post(
        f"{BASE_URL}/api/config/prompts",
        data={
            "agent_id": agent_id,
            "description": "System prompt for haiku_compose"
        },
        files={"file": f}
    )

print(f"Status: {r.status_code}")
result = r.json()
print(json.dumps(result, indent=2))

prompt_id = result.get("id")
print(f"\nSaved prompt_id = {prompt_id}")

In [ ]:
# Verify: get the agent by ID - should now include the prompt
r = requests.get(f"{BASE_URL}/api/config/agents/{agent_id}")
print(json.dumps(r.json(), indent=2))

# Look for the "prompt" field - it should show the blob_path

## 2.3 Enable the Timer Trigger Hook

The haiku agent is triggered by a **timer** that fires every minute. The code is already in the repo but **commented out**.

### What to do

1. Open `hooks/hooks.py` in VS Code
2. Find the commented-out block (lines 52-77)
3. **Uncomment** the entire `haiku_timer_pull` function

The uncommented code should look like this:

```python
@bp.timer_trigger(schedule="0 */1 * * * *", arg_name="timer", run_on_startup=False)
def haiku_timer_pull(timer: func.TimerRequest) -> None:
    """
    Timer trigger for periodic haiku generation.
    Runs every 1 minute.
    """
    logging.info("Haiku timer trigger fired")

    if timer.past_due:
        logging.warning("Timer is past due - running anyway")

    # Create trigger message with interval info
    message = {
        "source": "haiku_timer_trigger",
        "agent_type": "haiku_compose",
        "triggered_at": datetime.utcnow().isoformat(),
        "interval_minutes": 1  # Must match timer schedule
    }

    # Publish to Service Bus
    queue_name = "agent-workflow"
    result = publish_to_service_bus(queue_name, message, ensure_queue=False)

    logging.info(f"Timer trigger queued: {result.get('status')}")
```

### Key things to notice

| Element | Value | Meaning |
|---|---|---|
| `schedule` | `"0 */1 * * * *"` | Every 1 minute (NCRONTAB: `{sec} {min} {hr} {day} {mon} {dow}`) |
| `queue_name` | `"agent-workflow"` | Publishes directly to the agent queue (skips Gmail queue) |
| `agent_type` | `"haiku_compose"` | Tells the orchestrator which agent to run |
| `triggered_at` | UTC timestamp | Used by the haiku agent to determine even/odd minute |

### After editing
1. **Save the file** (`Cmd+S`)
2. **Restart the app** (`Ctrl+C` then `func start`, or `Shift+F5` then `F5` for debugger)

## 2.4 Test the Haiku Agent

After restarting the app, the timer fires within 1 minute.

### What to watch in the terminal

```
Haiku timer trigger fired
Timer trigger queued: success
...
Running agent: haiku_compose
```

### End-to-end flow

```
Timer fires (every 1 min)
    |
    v
Message published to "agent-workflow" queue
    |
    v
Queue consumer starts durable orchestration
    |
    v
Orchestrator calls haiku_compose agent
    |
    v
Agent checks triggered_at -> even or odd minute?
    |
    v
Composes haiku, returns JSON with next_action
    |
    v
Orchestrator routes to notification_content agent
    |
    v
notification_content formats haiku as HTML email
    |
    v
Email sent to NUS_EMAIL address
```

### Check your inbox
You should receive a haiku email within 2-3 minutes.

### Troubleshooting

| Problem | Likely Cause |
|---|---|
| No timer fires | Function not uncommented correctly, or app not restarted |
| Agent fails / error in terminal | Credential issues (`AZURE_*` env vars) or model not deployed |
| No email arrives | `GMAIL_*` or `NUS_EMAIL` env vars not set correctly |

---
# Part 3: Build the Knowledge Agent

The knowledge agent answers questions by searching uploaded documents using **RAG** (Retrieval-Augmented Generation).

**Characteristics:**
- Triggered when `email_triage` classifies an email as a `knowledge_question`
- Uses `file_search` tool (built into Azure AI) - no custom tool mappings needed
- Requires a `knowledge_source` with uploaded documents
- Returns answers with citations and confidence scores

**What we need to configure:**
1. Agent definition (with `knowledge_source`)
2. System prompt (instruction file)
3. Knowledge base documents (PDF, MD, TXT, DOCX)

## 3.1 Create the Agent Definition

Unlike the haiku agent, this agent needs a `knowledge_source` field. This tells the platform where to find documents in Blob Storage.

In [ ]:
# Create the knowledge_qa agent
r = requests.post(f"{BASE_URL}/api/config/agents", json={
    "name": "knowledge_qa",
    "description": "Knowledge base Q&A agent - answers questions using uploaded documents",
    "model": "gpt-4o-mini",
    "knowledge_source": "knowledge/knowledge_qa"
})

print(f"Status: {r.status_code}")
result = r.json()
print(json.dumps(result, indent=2))

# Save the agent ID
agent_id = result.get("id")
print(f"\nSaved agent_id = {agent_id}")

> **Note:** The `knowledge_source` value `"knowledge/knowledge_qa"` means documents will be stored under `knowledge/knowledge_qa/` in the `agent-config` blob container.

## 3.2 Upload the System Prompt

The instruction file is at `agents/instructions/knowledge_qa.system.md`. Let's inspect it first, then upload.

In [ ]:
# Inspect the knowledge_qa prompt
with open("agents/instructions/knowledge_qa.system.md", "r") as f:
    print(f.read())

In [ ]:
# Upload the prompt file
with open("agents/instructions/knowledge_qa.system.md", "rb") as f:
    r = requests.post(
        f"{BASE_URL}/api/config/prompts",
        data={
            "agent_id": agent_id,
            "description": "System prompt for knowledge_qa"
        },
        files={"file": f}
    )

print(f"Status: {r.status_code}")
result = r.json()
print(json.dumps(result, indent=2))

prompt_id = result.get("id")
print(f"\nSaved prompt_id = {prompt_id}")

## 3.3 Upload Knowledge Base Documents

Now we upload the documents the agent will search through. Supported formats: **PDF, TXT, MD, DOCX**.

Update the `file_path` variable below to point to your document.

In [ ]:
# ============================================
# UPDATE THIS PATH to your actual document
# ============================================
file_path = "path/to/your/document.pdf"

with open(file_path, "rb") as f:
    r = requests.post(
        f"{BASE_URL}/api/config/agents/{agent_id}/knowledge",
        files={"file": f}
    )

print(f"Status: {r.status_code}")
print(json.dumps(r.json(), indent=2))

In [ ]:
# To upload additional documents, repeat with a different file_path:

# file_path = "path/to/another/document.md"
# with open(file_path, "rb") as f:
#     r = requests.post(
#         f"{BASE_URL}/api/config/agents/{agent_id}/knowledge",
#         files={"file": f}
#     )
# print(json.dumps(r.json(), indent=2))

In [ ]:
# Verify: list all knowledge files for this agent
r = requests.get(f"{BASE_URL}/api/config/agents/{agent_id}/knowledge")
print(json.dumps(r.json(), indent=2))

## 3.4 How Knowledge Indexing Works

When the knowledge agent runs for the **first time** after uploading documents, the platform automatically:

| Step | What happens |
|---|---|
| **1. Load** | Downloads all files from `knowledge/knowledge_qa/` in Blob Storage |
| **2. Hash** | Computes an MD5 manifest of all file contents |
| **3. Compare** | Checks if manifest matches what's stored in `AgentDefinition.file_manifest` |
| **4. Index** | If files changed (or first run): creates a new Azure AI **vector store** and uploads all files |
| **5. Store** | Saves the `vector_store_id` and `file_manifest` back to the database |
| **6. Attach** | Passes the vector store to the agent as a `ToolResource` so `file_search` can query it |

On subsequent runs, if no files changed, it **skips re-indexing** (fast path).

## 3.5 Verify the Full Agent Configuration

Let's check the complete agent setup - prompt, tools, and knowledge source.

In [ ]:
# Get the full agent configuration
r = requests.get(f"{BASE_URL}/api/config/agents/{agent_id}")
agent_config = r.json()
print(json.dumps(agent_config, indent=2))

# Check the key fields:
print("\n--- Quick Check ---")
print(f"Name:             {agent_config.get('name')}")
print(f"Knowledge Source: {agent_config.get('knowledge_source')}")
print(f"Vector Store ID:  {agent_config.get('vector_store_id')}  (null until first run)")
print(f"Prompt:           {agent_config.get('prompt', {}).get('blob_path', 'MISSING')}")
print(f"Tools:            {agent_config.get('tools', [])}  (empty - file_search is built-in)")

## 3.6 Test the Knowledge Agent

The knowledge agent is triggered when `email_triage` classifies an incoming email as a `knowledge_question`.

### Option A: Send a test email (full pipeline)

1. Send an email **to the address configured in `NUS_EMAIL`** with a question about content in your uploaded documents
   - Example subject: `Question about DBA5115`
   - Example body: `What are the learning objectives for this course?`
2. Wait for the Gmail timer to pick it up (every 2 minutes), **or** trigger manually:

In [ ]:
# Manually trigger Gmail pull (instead of waiting for timer)
r = requests.post(f"{BASE_URL}/api/hooks/gmail_pull")
print(f"Status: {r.status_code}")
print(json.dumps(r.json(), indent=2))

### Full pipeline flow

```
Email arrives at NUS_EMAIL
    |
    v
Gmail timer (or manual pull) detects unread email
    |
    v
email_triage agent classifies as "knowledge_question"
    |
    v
Routes to knowledge_qa agent
    |
    v
knowledge_qa searches documents using file_search
    |
    v
Returns answer with citations + routes to notification_content
    |
    v
notification_content formats and sends email reply
```

### What to watch in the terminal

```
Running agent: knowledge_qa
Syncing knowledge for agent: knowledge_qa
Loading documents from blob: knowledge/knowledge_qa
Creating vector store: knowledge_qa
Uploading 1 files to vector store...
Agent response received
```

### Check your inbox
You should receive an email reply with the agent's answer, citing the source documents.

---
# Part 4: Review - What We Built

## Architecture Overview

```
                    Timer (1 min)
                         |
                         v
                  +-------------+
                  |   hooks.py  |  <-- Timer triggers & HTTP endpoints
                  +-------------+
                         |
                    Service Bus
                   "agent-workflow"
                         |
                         v
                  +-------------+
                  |  queues.py  |  <-- Queue consumers
                  +-------------+
                         |
                         v
                  +-----------------+
                  |  Orchestrator   |  <-- Durable Functions
                  +-----------------+
                    |           |
          +---------+           +---------+
          v                               v
  +---------------+             +------------------+
  | haiku_compose |             |   knowledge_qa   |
  | (no tools)    |             | (file_search)    |
  +---------------+             +------------------+
          |                               |
          +--------->  next_action  <-----+
                         |
                         v
              +---------------------+
              | notification_content|  <-- Formats & sends email
              +---------------------+
```

## Agents Summary

| Agent | Trigger | Tools | Knowledge | Purpose |
|---|---|---|---|---|
| `email_triage` | Gmail inbox | None | No | Classify and route emails |
| `notification_content` | Agent routing | `determine_recipient`, `send_email_notification` | No | Format and send email notifications |
| `haiku_compose` | Timer (1 min) | None | No | Compose haikus based on time |
| `knowledge_qa` | Email triage routing | `file_search` (built-in) | Yes | Answer questions from knowledge base |

## API Endpoints Used

| Step | Method | Endpoint | Purpose |
|---|---|---|---|
| Health Check | `GET` | `/api/health` | Verify app is running |
| List Agents | `GET` | `/api/config/agents` | See all registered agents |
| Create Agent | `POST` | `/api/config/agents` | Register agent in database |
| Verify Agent | `GET` | `/api/config/agents/{id}` | Check full agent config |
| Upload Prompt | `POST` | `/api/config/prompts` | Upload instruction .md file |
| Upload Knowledge | `POST` | `/api/config/agents/{id}/knowledge` | Upload documents for RAG |
| List Knowledge | `GET` | `/api/config/agents/{id}/knowledge` | List uploaded documents |
| Gmail Pull | `POST` | `/api/hooks/gmail_pull` | Manually trigger inbox check |

---
# Appendix: Cleanup

To delete an agent and all its associated resources (prompt, tools, vector store), use the cascade delete endpoint.

In [ ]:
# ============================================
# CAREFUL: This permanently deletes the agent!
# Set the ID of the agent you want to delete.
# ============================================

# delete_agent_id = agent_id  # or set a specific ID

# r = requests.delete(f"{BASE_URL}/api/config/agents/{delete_agent_id}")
# print(f"Status: {r.status_code}")
# print(json.dumps(r.json(), indent=2))

# Cascade deletes:
# - Prompt from database and Blob Storage
# - Tool mappings from database (blob only if not shared with other agents)
# - Vector store from Azure AI Foundry (if it exists)